# Lab 01: Implement a Simple Generative Algorithm for Data Generation

## Objective
Generate synthetic image data using a pre-trained generative AI model and evaluate classification performance.

### Experiment Overview
| Step | Description |
|------|-------------|
| 1 | Generate synthetic cat-species images using **Stable Diffusion XL Turbo** (40 species × 10 images = 400 images) |
| 2 | Evaluate a pre-trained **ResNet-18** on the generated dataset |
| 3 | Fine-tune **ResNet-18** on the synthetic dataset and measure accuracy |
| 4 | Build a **Custom CNN** from scratch and measure accuracy |
| 5 | **Zero-Shot Classification** using CLIP |
| 6 | **Few-Shot Classification** (5-shot) using feature extraction |

**Domain:** Animal Image Data (Cat Species)
**Generative Model:** Stable Diffusion XL Turbo (stabilityai/sdxl-turbo)
**Runtime:** Google Colab with T4 GPU (~30-40 min total)

## Step 0: Install Required Libraries
> ⚠️ **Run this cell first — restart runtime if prompted, then continue from the next cell.**

In [ ]:
# Install libraries (corrected package names for Colab)
!pip install -q diffusers transformers accelerate safetensors torch torchvision \
    open-clip-torch Pillow matplotlib scikit-learn tqdm sentencepiece protobuf huggingface_hub

In [ ]:
import os, random, warnings, gc
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms as T
import torchvision.models as models
from torchvision.datasets import ImageFolder

from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings("ignore")

# ---------- reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------- Device Selection (Colab Optimized) ----------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    try:
        # standard attribute
        mem = torch.cuda.get_device_properties(0).total_memory
    except AttributeError:
        # fallback for some older pytorch versions
        mem = torch.cuda.get_device_properties(0).total_mem
        
    print(f"   Memory: {mem / 1e9:.1f} GB")
else:
    if torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
        print("⚠️ GPU (CUDA) not found. Using Apple MPS.")
    else:
        DEVICE = torch.device("cpu")
        print("❌ NO GPU DETECTED! Image generation will be extremely slow (hours).")
        print("👉 If running on Google Colab: Go to Runtime > Change runtime type > Select T4 GPU.")
    
print(f"Using device: {DEVICE}")

## Step 1: Define 40 Cat Species & Input Prompts

We select **40 real-world cat species** and craft descriptive prompts that Stable Diffusion will use to generate photorealistic images.

In [ ]:
# ---------- 40 cat species ----------
CAT_SPECIES = [
    "Persian Cat", "Siamese Cat", "Maine Coon", "Bengal Cat", "Ragdoll Cat",
    "British Shorthair", "Abyssinian Cat", "Sphynx Cat", "Scottish Fold", "Birman Cat",
    "Russian Blue", "Norwegian Forest Cat", "Savannah Cat", "Burmese Cat", "Tonkinese Cat",
    "Oriental Shorthair", "Devon Rex", "Cornish Rex", "Himalayan Cat", "Exotic Shorthair",
    "American Shorthair", "Turkish Angora", "Manx Cat", "Chartreux Cat", "Balinese Cat",
    "Somali Cat", "Singapura Cat", "Egyptian Mau", "Ocicat", "Bombay Cat",
    "Korat Cat", "Havana Brown Cat", "Snowshoe Cat", "Ragamuffin Cat", "LaPerm Cat",
    "Selkirk Rex", "Toyger Cat", "Munchkin Cat", "Siberian Cat", "Turkish Van Cat",
]

NUM_SPECIES = len(CAT_SPECIES)
IMAGES_PER_SPECIES = 10
TOTAL_IMAGES = NUM_SPECIES * IMAGES_PER_SPECIES

# ---------- prompt templates for variation ----------
PROMPT_TEMPLATES = [
    "A high-quality photograph of a {species}, full body, natural lighting, sharp focus, 8k",
    "A professional studio portrait of a {species}, detailed fur texture, bokeh background",
    "A {species} sitting elegantly, photorealistic, award-winning wildlife photography",
    "A cute {species} looking at the camera, soft natural light, detailed, 4k photo",
    "A {species} in a home environment, cozy setting, natural pose, high resolution photo",
    "A beautiful {species} outdoors in a garden, golden hour lighting, DSLR quality",
    "Close-up portrait of a {species}, expressive eyes, fine fur detail, professional photo",
    "A {species} resting peacefully, warm tones, shallow depth of field, magazine quality",
    "An adorable {species} playing, action shot, sharp details, vibrant colors, 8k",
    "A majestic {species}, side profile, dramatic lighting, National Geographic style",
]

print(f"✅ {NUM_SPECIES} cat species defined")
print(f"✅ {len(PROMPT_TEMPLATES)} prompt templates ready")
print(f"✅ Total images to generate: {TOTAL_IMAGES}")
print(f"\n--- Sample Species ---")
for i, sp in enumerate(CAT_SPECIES[:5], 1):
    print(f"  {i}. {sp}")
print(f"  ... and {NUM_SPECIES - 5} more")

print(f"\n--- Sample Prompts (for '{CAT_SPECIES[0]}') ---")
for i, tmpl in enumerate(PROMPT_TEMPLATES[:5], 1):
    print(f"  {i}. {tmpl.format(species=CAT_SPECIES[0])}")

## Step 2: Generate Synthetic Images using Stable Diffusion XL Turbo

We use the **SDXL-Turbo** model from Stability AI — it produces high-quality 512×512 images in just **4 inference steps**, making it ideal for generating a large dataset quickly.

> ⏱ **Estimated time:** ~20-30 minutes for 400 images on a T4 GPU.

In [ ]:
from diffusers import DiffusionPipeline

# ---------- Load SDXL-Turbo pipeline ----------
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe = pipe.to(DEVICE)

# Disable safety checker to avoid false positives on animal images
if hasattr(pipe, 'safety_checker'):
    pipe.safety_checker = None
print("✅ SDXL-Turbo model loaded successfully!")

In [ ]:
# ---------- Generate images and save to disk ----------
OUTPUT_DIR = Path("synthetic_cat_dataset")
IMG_SIZE = 512  # SDXL-Turbo native resolution

for species_idx, species in enumerate(CAT_SPECIES):
    # Create a folder per species (class label)
    safe_name = species.lower().replace(" ", "_")
    species_dir = OUTPUT_DIR / safe_name
    species_dir.mkdir(parents=True, exist_ok=True)

    existing = list(species_dir.glob("*.png"))
    if len(existing) >= IMAGES_PER_SPECIES:
        print(f"[{species_idx+1:02d}/{NUM_SPECIES}] {species}: already generated ✔")
        continue

    print(f"[{species_idx+1:02d}/{NUM_SPECIES}] Generating {IMAGES_PER_SPECIES} images for: {species}")

    # Add tqdm for inner loop visibility
    for img_idx in tqdm(range(IMAGES_PER_SPECIES), desc=f"  Generating {species}", leave=False):
        prompt = PROMPT_TEMPLATES[img_idx].format(species=species)
        with torch.no_grad():
            image = pipe(
                prompt=prompt,
                num_inference_steps=4,
                guidance_scale=0.0,        # SDXL-Turbo uses guidance_scale=0
                height=IMG_SIZE,
                width=IMG_SIZE,
            ).images[0]

        save_path = species_dir / f"{safe_name}_{img_idx+1:02d}.png"
        image.save(save_path)

    # Clear CUDA cache periodically (if using CUDA)
    if DEVICE.type == "cuda" and (species_idx + 1) % 10 == 0:
        torch.cuda.empty_cache()
    if DEVICE.type == "mps":
         torch.mps.empty_cache()

print(f"\n✅ All {TOTAL_IMAGES} images generated and saved to '{OUTPUT_DIR}/'")

# Verify counts
total_saved = sum(1 for _ in OUTPUT_DIR.rglob("*.png"))
print(f"✅ Verified: {total_saved} images across {len(list(OUTPUT_DIR.iterdir()))} species folders")

In [ ]:
# ---------- Free GPU memory: unload the diffusion model ----------
del pipe
torch.cuda.empty_cache()
gc.collect()
print("✅ Diffusion model unloaded — GPU memory freed for classification tasks")

## Step 3: Display Sample Outputs

Let's visualize a grid of generated images — one sample from each of the 40 species.

In [ ]:
# ---------- Display one sample image per species ----------
OUTPUT_DIR = Path("synthetic_cat_dataset")
species_dirs = sorted([d for d in OUTPUT_DIR.iterdir() if d.is_dir()])

fig, axes = plt.subplots(5, 8, figsize=(24, 16))
fig.suptitle("Synthetic Cat Dataset — 1 Sample per Species (40 Species)", fontsize=18, y=1.01)

for idx, (ax, sp_dir) in enumerate(zip(axes.flat, species_dirs)):
    img_path = sorted(sp_dir.glob("*.png"))[0]
    img = Image.open(img_path)
    ax.imshow(img)
    title = sp_dir.name.replace("_", " ").title()
    ax.set_title(title, fontsize=8, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.savefig("sample_outputs_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Sample grid saved to 'sample_outputs_grid.png'")

## Step 4: Prepare Dataset for Classification

We split the generated dataset into **train (70%)** and **test (30%)** sets, apply standard ImageNet transforms, and create DataLoaders.

In [ ]:
# ---------- Dataset & DataLoader Setup ----------
OUTPUT_DIR = Path("synthetic_cat_dataset")

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Load full dataset to get labels and indices
full_dataset = ImageFolder(root=str(OUTPUT_DIR), transform=train_transform)
class_names = full_dataset.classes
num_classes = len(class_names)

# ---------- Train / Test split (70/30, stratified) ----------
from sklearn.model_selection import train_test_split

targets = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))
train_idx, test_idx = train_test_split(
    indices, test_size=0.3, random_state=SEED, stratify=targets
)

# Create subsets
train_dataset = Subset(full_dataset, train_idx)
# Test set uses test_transform (no augmentation)
test_full = ImageFolder(root=str(OUTPUT_DIR), transform=test_transform)
test_dataset = Subset(test_full, test_idx)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Dataset prepared!")
print(f"   Classes       : {num_classes}")
print(f"   Train samples : {len(train_dataset)}")
print(f"   Test samples  : {len(test_dataset)}")
print(f"   Batch size    : {BATCH_SIZE}")
print(f"\n   Sample class names: {class_names[:5]} ...")

## Step 5: Classification with Pre-trained ResNet-18

### 5A — Zero-shot ResNet (No fine-tuning)
First, we evaluate ResNet-18 with ImageNet weights directly on our cat dataset. Since ImageNet contains some cat-related classes, it can make *some* predictions, but it doesn't know our 40 specific species labels — so accuracy will be low.

In [ ]:
# ---------- 5A: ResNet-18 WITHOUT fine-tuning (Zero-Shot on ImageNet classes) ----------
resnet_pretrained = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet_pretrained = resnet_pretrained.to(DEVICE)
resnet_pretrained.eval()

# ImageNet has 1000 classes — map our 40 classes to nearest ImageNet cat class
# We'll just see what ImageNet predicts and compare feature quality
imagenet_cat_class_ids = list(range(281, 294))  # ImageNet cat breed classes (281-293)

correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = resnet_pretrained(images)
        # Check if predicted class falls in cat-related ImageNet classes (281-293)
        _, predicted = outputs.max(1)
        # Map: if prediction is any cat class → mark as "detected cat"
        for pred, label in zip(predicted.cpu().numpy(), labels.numpy()):
            all_preds.append(pred)
            all_labels.append(label)
            total += 1

# Since ImageNet ResNet cannot predict our 40 custom classes, we report
# what percentage of images it at least identifies as "cat"
cat_detected = sum(1 for p in all_preds if p in imagenet_cat_class_ids)
cat_detection_rate = cat_detected / total * 100

print("=" * 60)
print("ResNet-18 (Pre-trained, NO fine-tuning) — Zero-Shot Evaluation")
print("=" * 60)
print(f"Total test images     : {total}")
print(f"Detected as cat (ImageNet cat classes 281-293): {cat_detected}/{total}")
print(f"Cat detection rate    : {cat_detection_rate:.1f}%")
print(f"\n⚠️  Note: The pre-trained ResNet-18 only knows 1000 ImageNet classes,")
print(f"   NOT our 40 specific cat species. It cannot classify species directly.")
print(f"   Fine-tuning is required for species-level classification.")

### 5B — Fine-tuned ResNet-18

Now we **replace the final classification layer** with a 40-class head and fine-tune the model on our synthetic dataset.

In [ ]:
# ---------- Helper: Training & Evaluation functions ----------
def train_model(model, train_loader, criterion, optimizer, num_epochs, model_name="Model"):
    """Train the model and return loss/accuracy history."""
    model.train()
    history = {"train_loss": [], "train_acc": []}

    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            pbar.set_postfix(loss=loss.item(), acc=f"{100.*correct/total:.1f}%")

        epoch_loss = running_loss / total
        epoch_acc = 100. * correct / total
        history["train_loss"].append(epoch_loss)
        history["train_acc"].append(epoch_acc)
        print(f"  [{model_name}] Epoch {epoch+1}/{num_epochs} — Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2f}%")

    return history


def evaluate_model(model, test_loader, class_names, model_name="Model"):
    """Evaluate the model and print classification report."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    accuracy = accuracy_score(all_labels, all_preds) * 100
    print(f"\n{'='*60}")
    print(f"{model_name} — Test Accuracy: {accuracy:.2f}%")
    print(f"{'='*60}")
    # Show per-class report (abbreviated for readability)
    report = classification_report(
        all_labels, all_preds,
        target_names=[n.replace("_", " ").title() for n in class_names],
        zero_division=0,
        output_dict=True,
    )
    # Print top-5 and bottom-5 classes by f1-score
    class_scores = {k: v["f1-score"] for k, v in report.items()
                    if k not in ("accuracy", "macro avg", "weighted avg")}
    sorted_classes = sorted(class_scores.items(), key=lambda x: x[1], reverse=True)

    print(f"\n📊 Macro Avg F1: {report['macro avg']['f1-score']:.4f}")
    print(f"📊 Weighted Avg F1: {report['weighted avg']['f1-score']:.4f}")
    print(f"\n  Top-5 classes:")
    for name, f1 in sorted_classes[:5]:
        print(f"    {name}: F1={f1:.2f}")
    print(f"\n  Bottom-5 classes:")
    for name, f1 in sorted_classes[-5:]:
        print(f"    {name}: F1={f1:.2f}")

    return accuracy, all_preds, all_labels

print("✅ Training & evaluation helper functions defined")

In [ ]:
# ---------- 5B: Fine-tune ResNet-18 ----------
# Load pre-trained ResNet-18 and replace final FC layer
resnet_ft = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze early layers (only train last 2 blocks + FC)
for name, param in resnet_ft.named_parameters():
    if "layer3" not in name and "layer4" not in name and "fc" not in name:
        param.requires_grad = False

# Replace classifier head
resnet_ft.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(resnet_ft.fc.in_features, num_classes),
)
resnet_ft = resnet_ft.to(DEVICE)

# Training config
NUM_EPOCHS_RESNET = 15
criterion = nn.CrossEntropyLoss()
optimizer_resnet = optim.Adam(
    filter(lambda p: p.requires_grad, resnet_ft.parameters()),
    lr=1e-3, weight_decay=1e-4
)
scheduler_resnet = optim.lr_scheduler.StepLR(optimizer_resnet, step_size=5, gamma=0.5)

print(f"🏋️ Fine-tuning ResNet-18 for {NUM_EPOCHS_RESNET} epochs...")
print(f"   Trainable params: {sum(p.numel() for p in resnet_ft.parameters() if p.requires_grad):,}")
print(f"   Total params    : {sum(p.numel() for p in resnet_ft.parameters()):,}\n")

resnet_history = train_model(
    resnet_ft, train_loader, criterion, optimizer_resnet,
    NUM_EPOCHS_RESNET, model_name="ResNet-18 (Fine-tuned)"
)

# Step the scheduler
for _ in range(NUM_EPOCHS_RESNET):
    scheduler_resnet.step()

In [ ]:
# ---------- Evaluate Fine-tuned ResNet-18 ----------
resnet_acc, resnet_preds, resnet_labels = evaluate_model(
    resnet_ft, test_loader, class_names, model_name="ResNet-18 (Fine-tuned)"
)

## Step 6: Custom CNN Model (Built from Scratch)

We build a custom CNN with 4 convolutional blocks followed by fully connected layers, trained entirely from scratch on the synthetic dataset.

In [ ]:
# ---------- Custom CNN Architecture ----------
class CustomCatCNN(nn.Module):
    """
    Custom CNN for cat species classification.
    Architecture: 4 Conv blocks → Global Avg Pool → FC layers
    """
    def __init__(self, num_classes=40):
        super(CustomCatCNN, self).__init__()

        # Block 1: 3 → 32
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),       # 224 → 112
            nn.Dropout2d(0.25),
        )

        # Block 2: 32 → 64
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),       # 112 → 56
            nn.Dropout2d(0.25),
        )

        # Block 3: 64 → 128
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),       # 56 → 28
            nn.Dropout2d(0.25),
        )

        # Block 4: 128 → 256
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),  # Global Average Pooling
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.classifier(x)
        return x


# Instantiate
custom_cnn = CustomCatCNN(num_classes=num_classes).to(DEVICE)

total_params = sum(p.numel() for p in custom_cnn.parameters())
trainable_params = sum(p.numel() for p in custom_cnn.parameters() if p.requires_grad)

print("✅ Custom CNN architecture:")
print(custom_cnn)
print(f"\n   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable_params:,}")

In [ ]:
# ---------- Train Custom CNN ----------
NUM_EPOCHS_CUSTOM = 25
criterion = nn.CrossEntropyLoss()
optimizer_custom = optim.Adam(custom_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_custom = optim.lr_scheduler.CosineAnnealingLR(optimizer_custom, T_max=NUM_EPOCHS_CUSTOM)

print(f"🏋️ Training Custom CNN for {NUM_EPOCHS_CUSTOM} epochs...\n")

custom_history = train_model(
    custom_cnn, train_loader, criterion, optimizer_custom,
    NUM_EPOCHS_CUSTOM, model_name="Custom CNN"
)

for _ in range(NUM_EPOCHS_CUSTOM):
    scheduler_custom.step()

In [ ]:
# ---------- Evaluate Custom CNN ----------
custom_acc, custom_preds, custom_labels = evaluate_model(
    custom_cnn, test_loader, class_names, model_name="Custom CNN"
)

## Step 6B: Detailed Comparison — ResNet-18 (Fine-tuned) vs Custom CNN

A head-to-head comparison of the two supervised models trained on our synthetic cat dataset.

In [ ]:
# ---------- Architecture Comparison Table ----------
resnet_total = sum(p.numel() for p in resnet_ft.parameters())
resnet_trainable = sum(p.numel() for p in resnet_ft.parameters() if p.requires_grad)
custom_total = sum(p.numel() for p in custom_cnn.parameters())
custom_trainable = sum(p.numel() for p in custom_cnn.parameters() if p.requires_grad)

print("=" * 75)
print("       ARCHITECTURE COMPARISON: ResNet-18 vs Custom CNN")
print("=" * 75)
print(f"{'Property':<35} {'ResNet-18 (FT)':>18} {'Custom CNN':>18}")
print("-" * 75)
print(f"{'Pre-trained weights':<35} {'✅ ImageNet':>18} {'❌ None':>18}")
print(f"{'Total parameters':<35} {resnet_total:>15,}   {custom_total:>15,}")
print(f"{'Trainable parameters':<35} {resnet_trainable:>15,}   {custom_trainable:>15,}")
print(f"{'Frozen parameters':<35} {resnet_total - resnet_trainable:>15,}   {0:>15,}")
print(f"{'Training epochs':<35} {NUM_EPOCHS_RESNET:>18} {NUM_EPOCHS_CUSTOM:>18}")
print(f"{'Conv blocks':<35} {'4 (ResNet blocks)':>18} {'4 (Custom)':>18}")
print(f"{'Pooling':<35} {'Avg + FC':>18} {'Global Avg Pool':>18}")
print(f"{'Regularization':<35} {'Dropout(0.3)':>18} {'Dropout+BN':>18}")
print(f"{'Optimizer':<35} {'Adam':>18} {'Adam':>18}")
print(f"{'Learning rate':<35} {'1e-3':>18} {'1e-3':>18}")
print(f"{'LR Scheduler':<35} {'StepLR':>18} {'CosineAnnealing':>18}")
print(f"{'Test Accuracy':<35} {resnet_acc:>17.2f}% {custom_acc:>17.2f}%")
print("=" * 75)
winner = "ResNet-18 (Fine-tuned)" if resnet_acc > custom_acc else "Custom CNN"
diff = abs(resnet_acc - custom_acc)
print(f"\n🏆 Winner: {winner} (by {diff:.2f}% margin)")

In [ ]:
# ---------- Side-by-side Training Curves ----------
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# --- Loss Comparison ---
ax = axes[0]
epochs_r = range(1, NUM_EPOCHS_RESNET + 1)
epochs_c = range(1, NUM_EPOCHS_CUSTOM + 1)
ax.plot(epochs_r, resnet_history["train_loss"], "b-o", label="ResNet-18 (Fine-tuned)", markersize=4)
ax.plot(epochs_c, custom_history["train_loss"], "r-s", label="Custom CNN", markersize=4)
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Training Loss", fontsize=12)
ax.set_title("Training Loss: ResNet-18 vs Custom CNN", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# --- Accuracy Comparison ---
ax = axes[1]
ax.plot(epochs_r, resnet_history["train_acc"], "b-o", label="ResNet-18 (Fine-tuned)", markersize=4)
ax.plot(epochs_c, custom_history["train_acc"], "r-s", label="Custom CNN", markersize=4)
# Add horizontal lines for test accuracy
ax.axhline(y=resnet_acc, color="blue", linestyle="--", alpha=0.5, label=f"ResNet-18 Test Acc ({resnet_acc:.1f}%)")
ax.axhline(y=custom_acc, color="red", linestyle="--", alpha=0.5, label=f"Custom CNN Test Acc ({custom_acc:.1f}%)")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("Training Accuracy: ResNet-18 vs Custom CNN", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("resnet_vs_custom_training.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Training comparison saved to 'resnet_vs_custom_training.png'")

In [ ]:
# ---------- Confusion Matrices Side-by-Side ----------
from sklearn.metrics import confusion_matrix
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
short_names = [n.replace("_cat", "").replace("_", " ").title()[:12] for n in class_names]

for ax, preds, labels, title, cmap in [
    (axes[0], resnet_preds, resnet_labels, "ResNet-18 (Fine-tuned)", "Blues"),
    (axes[1], custom_preds, custom_labels, "Custom CNN (From Scratch)", "Reds"),
]:
    cm = confusion_matrix(labels, preds, labels=range(num_classes))
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap, aspect="auto")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True", fontsize=11)
    ax.set_xticks(range(num_classes))
    ax.set_yticks(range(num_classes))
    ax.set_xticklabels(short_names, rotation=90, fontsize=5)
    ax.set_yticklabels(short_names, fontsize=5)
    fig.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Confusion Matrices — ResNet-18 vs Custom CNN", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrices saved to 'confusion_matrices_comparison.png'")

In [ ]:
# ---------- Per-Class F1-Score Comparison ----------
from sklearn.metrics import f1_score, precision_score, recall_score

resnet_f1 = f1_score(resnet_labels, resnet_preds, labels=range(num_classes), average=None, zero_division=0)
custom_f1 = f1_score(custom_labels, custom_preds, labels=range(num_classes), average=None, zero_division=0)

resnet_precision = precision_score(resnet_labels, resnet_preds, labels=range(num_classes), average=None, zero_division=0)
custom_precision = precision_score(custom_labels, custom_preds, labels=range(num_classes), average=None, zero_division=0)

resnet_recall = recall_score(resnet_labels, resnet_preds, labels=range(num_classes), average=None, zero_division=0)
custom_recall = recall_score(custom_labels, custom_preds, labels=range(num_classes), average=None, zero_division=0)

# --- Bar chart: Per-class F1 comparison ---
fig, ax = plt.subplots(figsize=(18, 6))
x = np.arange(num_classes)
width = 0.35

bars1 = ax.bar(x - width/2, resnet_f1, width, label="ResNet-18 (Fine-tuned)", color="#3498db", alpha=0.85)
bars2 = ax.bar(x + width/2, custom_f1, width, label="Custom CNN", color="#e74c3c", alpha=0.85)

ax.set_xlabel("Cat Species", fontsize=12)
ax.set_ylabel("F1-Score", fontsize=12)
ax.set_title("Per-Class F1-Score: ResNet-18 vs Custom CNN", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(short_names, rotation=90, fontsize=6)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)
ax.grid(axis="y", alpha=0.3)

# Highlight classes where one model clearly wins
for i in range(num_classes):
    if resnet_f1[i] - custom_f1[i] > 0.3:
        ax.annotate("▲", (i - width/2, resnet_f1[i] + 0.02), ha="center", fontsize=8, color="blue")
    elif custom_f1[i] - resnet_f1[i] > 0.3:
        ax.annotate("▲", (i + width/2, custom_f1[i] + 0.02), ha="center", fontsize=8, color="red")

plt.tight_layout()
plt.savefig("per_class_f1_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Per-class F1 comparison saved to 'per_class_f1_comparison.png'")

In [ ]:
# ---------- Detailed Metrics Summary ----------
resnet_wins = sum(1 for r, c in zip(resnet_f1, custom_f1) if r > c)
custom_wins = sum(1 for r, c in zip(resnet_f1, custom_f1) if c > r)
ties = num_classes - resnet_wins - custom_wins

print("=" * 75)
print("     DETAILED METRICS: ResNet-18 (Fine-tuned) vs Custom CNN")
print("=" * 75)
print(f"\n{'Metric':<30} {'ResNet-18':>18} {'Custom CNN':>18}")
print("-" * 70)
print(f"{'Test Accuracy':<30} {resnet_acc:>17.2f}% {custom_acc:>17.2f}%")
print(f"{'Macro Avg Precision':<30} {np.mean(resnet_precision):>17.4f}  {np.mean(custom_precision):>17.4f}")
print(f"{'Macro Avg Recall':<30} {np.mean(resnet_recall):>17.4f}  {np.mean(custom_recall):>17.4f}")
print(f"{'Macro Avg F1-Score':<30} {np.mean(resnet_f1):>17.4f}  {np.mean(custom_f1):>17.4f}")
print(f"{'Weighted Avg F1-Score':<30} {f1_score(resnet_labels, resnet_preds, average=\"weighted\", zero_division=0):>17.4f}  {f1_score(custom_labels, custom_preds, average=\"weighted\", zero_division=0):>17.4f}")
print("-" * 70)
print(f"{'Per-class wins (F1)':<30} {resnet_wins:>18} {custom_wins:>18}")
print(f"{'Ties':<30} {ties:>18}")
print(f"{'Total parameters':<30} {resnet_total:>15,}   {custom_total:>15,}")
print(f"{'Trainable parameters':<30} {resnet_trainable:>15,}   {custom_trainable:>15,}")
print(f"{'Training epochs':<30} {NUM_EPOCHS_RESNET:>18} {NUM_EPOCHS_CUSTOM:>18}")
print("=" * 75)

# --- Top-5 classes where ResNet beats Custom CNN and vice versa ---
diff_f1 = resnet_f1 - custom_f1
sorted_diff_idx = np.argsort(diff_f1)

print(f"\n📊 Top-5 classes where ResNet-18 outperforms Custom CNN:")
for i in sorted_diff_idx[-5:][::-1]:
    name = class_names[i].replace("_", " ").title()
    print(f"   {name:<25} ResNet F1={resnet_f1[i]:.2f}  Custom F1={custom_f1[i]:.2f}  (Δ = +{diff_f1[i]:.2f})")

print(f"\n📊 Top-5 classes where Custom CNN outperforms ResNet-18:")
for i in sorted_diff_idx[:5]:
    name = class_names[i].replace("_", " ").title()
    print(f"   {name:<25} Custom F1={custom_f1[i]:.2f}  ResNet F1={resnet_f1[i]:.2f}  (Δ = +{-diff_f1[i]:.2f})")

print(f"\n💡 Analysis:")
if resnet_acc > custom_acc:
    print(f"   → ResNet-18 (Fine-tuned) outperforms Custom CNN overall.")
    print(f"   → Transfer learning from ImageNet gives a significant advantage,")
    print(f"     especially with limited training data (only {len(train_dataset)} images).")
    print(f"   → Pre-trained features capture low-level patterns (edges, textures)")
    print(f"     that would take much longer for a from-scratch model to learn.")
else:
    print(f"   → Custom CNN matches or beats ResNet-18 despite no pre-training!")
    print(f"   → This suggests the augmentation and architecture design were effective.")
print(f"   → With only {IMAGES_PER_SPECIES} images per class, transfer learning is")
print(f"     generally the better strategy for real-world applications.")

In [ ]:
# ---------- Radar Chart: Multi-metric comparison ----------
from matplotlib.patches import FancyBboxPatch

metrics_names = ["Accuracy", "Macro\nPrecision", "Macro\nRecall", "Macro\nF1", "Weighted\nF1"]
resnet_metrics = [
    resnet_acc / 100,
    np.mean(resnet_precision),
    np.mean(resnet_recall),
    np.mean(resnet_f1),
    f1_score(resnet_labels, resnet_preds, average="weighted", zero_division=0),
]
custom_metrics = [
    custom_acc / 100,
    np.mean(custom_precision),
    np.mean(custom_recall),
    np.mean(custom_f1),
    f1_score(custom_labels, custom_preds, average="weighted", zero_division=0),
]

# Radar chart
angles = np.linspace(0, 2 * np.pi, len(metrics_names), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

resnet_vals = resnet_metrics + resnet_metrics[:1]
custom_vals = custom_metrics + custom_metrics[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, resnet_vals, "b-o", linewidth=2, markersize=8, label="ResNet-18 (Fine-tuned)")
ax.fill(angles, resnet_vals, alpha=0.15, color="blue")
ax.plot(angles, custom_vals, "r-s", linewidth=2, markersize=8, label="Custom CNN")
ax.fill(angles, custom_vals, alpha=0.15, color="red")

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_names, fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.0)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8)
ax.set_title("Multi-Metric Radar: ResNet-18 vs Custom CNN", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("radar_resnet_vs_custom.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Radar chart saved to 'radar_resnet_vs_custom.png'")

## Step 7: Zero-Shot Classification using CLIP

**Zero-shot learning** means the model classifies images into categories it has **never been explicitly trained on**. We use OpenAI's CLIP model, which can match images to text descriptions.

In [ ]:
# ---------- Zero-Shot Classification with CLIP ----------
import open_clip

# Load CLIP model
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_model = clip_model.to(DEVICE)
clip_model.eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")

# Prepare text prompts for each species
text_labels = [f"a photo of a {species.lower()}" for species in CAT_SPECIES]
text_tokens = tokenizer(text_labels).to(DEVICE)

# Encode text features once
with torch.no_grad():
    text_features = clip_model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print(f"✅ CLIP model loaded (ViT-B-32)")
print(f"✅ {len(text_labels)} text prompts encoded")
print(f"\n   Sample prompts:")
for t in text_labels[:5]:
    print(f"     → \"{t}\"")

In [ ]:
# ---------- Run Zero-Shot classification on test set ----------
OUTPUT_DIR = Path("synthetic_cat_dataset")
test_full_clip = ImageFolder(root=str(OUTPUT_DIR))
# Map dataset class indices to our CAT_SPECIES order
dataset_classes = test_full_clip.classes  # sorted folder names

# Build mapping from dataset class index → CAT_SPECIES index
species_name_map = {s.lower().replace(" ", "_"): idx for idx, s in enumerate(CAT_SPECIES)}

clip_preds = []
clip_labels = []
correct = 0
total = 0

with torch.no_grad():
    for img_idx in test_idx:
        img_path, dataset_label = test_full_clip.samples[img_idx]
        # Get true species index
        folder_name = dataset_classes[dataset_label]
        true_label = species_name_map.get(folder_name, -1)
        if true_label == -1:
            continue

        # Preprocess image for CLIP
        img = Image.open(img_path).convert("RGB")
        img_tensor = clip_preprocess(img).unsqueeze(0).to(DEVICE)

        # Encode image
        image_features = clip_model.encode_image(img_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Compute similarity
        similarity = (image_features @ text_features.T).squeeze(0)
        pred_label = similarity.argmax().item()

        clip_preds.append(pred_label)
        clip_labels.append(true_label)

        if pred_label == true_label:
            correct += 1
        total += 1

clip_zero_shot_acc = correct / total * 100

print("=" * 60)
print("CLIP Zero-Shot Classification Results")
print("=" * 60)
print(f"Total test images : {total}")
print(f"Correct           : {correct}")
print(f"Accuracy          : {clip_zero_shot_acc:.2f}%")
print(f"\n💡 This is ZERO-SHOT: CLIP was never trained on our specific dataset!")
print(f"   It classifies purely by matching image-text similarity.")

## Step 8: Few-Shot Classification (5-Shot)

**Few-shot learning** uses only a **small number of labeled examples** (here, 5 per class) to classify new images. We use **CLIP features** + a **nearest-centroid classifier**: compute the mean feature vector for each class from 5 examples, then classify by finding the closest centroid.

In [ ]:
# ---------- Few-Shot Classification (5-shot) using CLIP embeddings ----------
K_SHOT = 5  # number of support examples per class

OUTPUT_DIR = Path("synthetic_cat_dataset")
full_dataset_clip = ImageFolder(root=str(OUTPUT_DIR))
dataset_classes = full_dataset_clip.classes

# Organize indices by class
from collections import defaultdict
class_indices = defaultdict(list)
for idx, (_, label) in enumerate(full_dataset_clip.samples):
    class_indices[label].append(idx)

# Split: K_SHOT for support, rest for query
support_indices = []
query_indices = []
for label, indices_list in class_indices.items():
    random.shuffle(indices_list)
    support_indices.extend(indices_list[:K_SHOT])
    query_indices.extend(indices_list[K_SHOT:])

print(f"✅ Few-Shot Setup ({K_SHOT}-shot):")
print(f"   Support set: {len(support_indices)} images ({K_SHOT} per class × {num_classes} classes)")
print(f"   Query set  : {len(query_indices)} images")

# ---------- Extract CLIP features for all images ----------
def extract_clip_features(indices, dataset, model, preprocess):
    """Extract CLIP image features for given indices."""
    features = []
    labels = []
    for idx in tqdm(indices, desc="Extracting CLIP features", leave=False):
        img_path, label = dataset.samples[idx]
        img = Image.open(img_path).convert("RGB")
        img_tensor = preprocess(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feat = model.encode_image(img_tensor)
            feat = feat / feat.norm(dim=-1, keepdim=True)
        features.append(feat.cpu().numpy().flatten())
        labels.append(label)
    return np.array(features), np.array(labels)

print("\n📐 Extracting features for support set...")
support_features, support_labels = extract_clip_features(
    support_indices, full_dataset_clip, clip_model, clip_preprocess
)

print("📐 Extracting features for query set...")
query_features, query_labels = extract_clip_features(
    query_indices, full_dataset_clip, clip_model, clip_preprocess
)

# ---------- Nearest Centroid Classifier ----------
# Compute class centroids from support set
centroids = []
for c in range(num_classes):
    mask = support_labels == c
    centroid = support_features[mask].mean(axis=0)
    centroid = centroid / np.linalg.norm(centroid)  # normalize
    centroids.append(centroid)
centroids = np.array(centroids)

# Classify query set
similarities = query_features @ centroids.T
few_shot_preds = similarities.argmax(axis=1)

few_shot_acc = accuracy_score(query_labels, few_shot_preds) * 100

print(f"\n{'='*60}")
print(f"Few-Shot Classification Results ({K_SHOT}-Shot)")
print(f"{'='*60}")
print(f"Support examples/class : {K_SHOT}")
print(f"Query images           : {len(query_labels)}")
print(f"Accuracy               : {few_shot_acc:.2f}%")
print(f"\n💡 Only {K_SHOT} labeled images per class were used for learning!")

## Step 9: Training Curves & Final Comparison

### 9A — Training Loss & Accuracy Curves

In [ ]:
# ---------- Plot Training Curves ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(resnet_history["train_loss"], "b-o", label="ResNet-18 (Fine-tuned)", markersize=4)
axes[0].plot(custom_history["train_loss"], "r-s", label="Custom CNN", markersize=4)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training Loss")
axes[0].set_title("Training Loss Over Epochs")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(resnet_history["train_acc"], "b-o", label="ResNet-18 (Fine-tuned)", markersize=4)
axes[1].plot(custom_history["train_acc"], "r-s", label="Custom CNN", markersize=4)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Training Accuracy (%)")
axes[1].set_title("Training Accuracy Over Epochs")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Training curves saved to 'training_curves.png'")

### 9B — Final Accuracy Comparison (All Methods)

In [ ]:
# ---------- Final Comparison Bar Chart ----------
methods = [
    "ResNet-18\n(Pre-trained,\nNo Fine-tune)",
    "ResNet-18\n(Fine-tuned)",
    "Custom CNN\n(From Scratch)",
    "CLIP\nZero-Shot",
    f"CLIP\nFew-Shot\n({K_SHOT}-shot)",
]
accuracies = [
    cat_detection_rate,   # ResNet pre-trained (cat detection rate)
    resnet_acc,           # ResNet fine-tuned
    custom_acc,           # Custom CNN
    clip_zero_shot_acc,   # CLIP zero-shot
    few_shot_acc,         # CLIP few-shot
]
colors = ["#e74c3c", "#2ecc71", "#3498db", "#9b59b6", "#f39c12"]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(methods, accuracies, color=colors, edgecolor="black", linewidth=0.8)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 1,
            f"{acc:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=12)

ax.set_ylabel("Accuracy (%)", fontsize=13)
ax.set_title("Cat Species Classification — Accuracy Comparison Across Methods", fontsize=14, fontweight="bold")
ax.set_ylim(0, max(accuracies) + 15)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Comparison chart saved to 'accuracy_comparison.png'")

In [ ]:
# ---------- Summary Table ----------
print("=" * 70)
print("         FINAL RESULTS SUMMARY — Cat Species Classification")
print("=" * 70)
print(f"{'Method':<40} {'Accuracy':>10} {'Type':>15}")
print("-" * 70)
print(f"{'ResNet-18 (Pre-trained, No Fine-tune)':<40} {cat_detection_rate:>9.2f}% {'Zero-Shot':>15}")
print(f"{'ResNet-18 (Fine-tuned)':<40} {resnet_acc:>9.2f}% {'Supervised':>15}")
print(f"{'Custom CNN (From Scratch)':<40} {custom_acc:>9.2f}% {'Supervised':>15}")
print(f"{'CLIP Zero-Shot':<40} {clip_zero_shot_acc:>9.2f}% {'Zero-Shot':>15}")
print(f"{'CLIP Few-Shot (5-shot)':<40} {few_shot_acc:>9.2f}% {'Few-Shot':>15}")
print("=" * 70)
print(f"\n📊 Dataset: {TOTAL_IMAGES} synthetic images across {NUM_SPECIES} cat species")
print(f"📊 Generated using: Stable Diffusion XL Turbo (stabilityai/sdxl-turbo)")
print(f"📊 Image size: {IMG_SIZE}×{IMG_SIZE} pixels")

## Conclusion

| Aspect | Details |
|--------|---------|
| **Domain** | Animal Image Data — 40 Cat Species |
| **Generative Model** | Stable Diffusion XL Turbo (`stabilityai/sdxl-turbo`) |
| **Dataset Size** | 400 images (40 species × 10 images each) |
| **Input Prompts** | 10 diverse prompt templates per species |
| **Classification Models** | ResNet-18 (pre-trained + fine-tuned), Custom CNN |
| **Zero-Shot** | CLIP ViT-B-32 — classifies without any training on our data |
| **Few-Shot** | CLIP embeddings + Nearest Centroid (5-shot per class) |

### Key Observations:
1. **ResNet-18 (pre-trained, no fine-tuning)** can detect that images are cats but cannot distinguish species — expected since it was trained on ImageNet's limited cat categories.
2. **ResNet-18 (fine-tuned)** achieves higher accuracy after adapting to our 40-class dataset with transfer learning.
3. **Custom CNN** trained from scratch shows that a bespoke architecture can also learn patterns, though typically lower accuracy than transfer learning with limited data.
4. **CLIP Zero-Shot** demonstrates the power of vision-language models — it classifies species purely from text descriptions without any task-specific training.
5. **CLIP Few-Shot (5-shot)** improves over zero-shot by learning class centroids from just 5 examples per species.

> ✅ **Experiment Complete:** A synthetic cat-species dataset was successfully generated using a generative AI model and evaluated across multiple classification paradigms.